# T1.2 — Open-LLM structured-output spike

Verify `Qwen/Qwen2.5-3B-Instruct` produces valid-JSON tactic classifications
at >= 9/10 rate with labels drawn from the taxonomy in
`docs/ARCHITECTURE.md §3.3`.

If < 9/10 parses: fall back to `microsoft/Phi-3-mini-4k-instruct`, then
`google/gemma-2-2b-it`, then escalate to `Qwen2.5-7B-Instruct` in 4-bit
or constrained decoding via `outlines` (see docs/ARCHITECTURE.md §2.3).

In [ ]:
# --- VishGuard setup cell (copy from notebooks/00_colabSetup.ipynb) ---
import os, sys, subprocess
from pathlib import Path
REPO_URL = os.environ.get('VISHGUARD_REPO_URL', 'https://github.com/ben-blake/vishguard.git')
DRIVE_SRC = '/content/drive/MyDrive/vishguard'
REPO_DIR = Path('/content/vishguard')
ON_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
def sh(cmd, check=True):
    print('$', cmd); subprocess.run(cmd, shell=True, check=check)
if ON_COLAB:
    try:
        from google.colab import drive; drive.mount('/content/drive', force_remount=False)
    except Exception as e: print('Drive mount skipped:', e)
    if REPO_URL:
        if REPO_DIR.exists(): sh(f'cd {REPO_DIR} && git pull --ff-only')
        else: sh(f'git clone {REPO_URL} {REPO_DIR}')
    elif Path(DRIVE_SRC).exists():
        REPO_DIR.mkdir(parents=True, exist_ok=True)
        sh(f'rsync -a --delete --exclude .venv --exclude __pycache__ {DRIVE_SRC}/ {REPO_DIR}/')
    else:
        raise RuntimeError('Set VISHGUARD_REPO_URL or copy the repo to MyDrive/vishguard/')
    sh(f'pip install -q -e {REPO_DIR}')
    sh(f'pip install -q -r {REPO_DIR}/requirements.txt')
    import torch
    if torch.cuda.is_available():
        sh(f'pip install -q -r {REPO_DIR}/requirements-gpu.txt')
    src = str(REPO_DIR / 'src')
    if src not in sys.path: sys.path.insert(0, src)
import vishguard; print('vishguard', vishguard.__version__)

In [ ]:
import json, re, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from vishguard.types import TACTIC_TAXONOMY

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto' if torch.cuda.is_available() else None,
)
print('loaded:', MODEL_ID, 'device:', next(model.parameters()).device)

In [ ]:
TAXONOMY_LINE = ', '.join(TACTIC_TAXONOMY)
SYSTEM = (
    'You are a social-engineering analyst. Given a phone-call transcript, '
    'identify the persuasion tactics in use. You MUST output a JSON array '
    'and nothing else. Each element has fields: label (one of: '
    f'{TAXONOMY_LINE}), confidence (0.0-1.0), evidenceSpans (array of '
    'verbatim quotes from the transcript).'
)
FEW_SHOT = (
    'Example transcript: "This is the IRS. Pay 2400 in gift cards or be arrested today."\n'
    'Example JSON:\n'
    '[{"label": "authority", "confidence": 0.95, "evidenceSpans": ["This is the IRS"]},'
    ' {"label": "fear_intimidation", "confidence": 0.9, "evidenceSpans": ["be arrested today"]},'
    ' {"label": "financial_manipulation", "confidence": 0.9, "evidenceSpans": ["2400 in gift cards"]},'
    ' {"label": "urgency", "confidence": 0.8, "evidenceSpans": ["today"]}]\n'
)

def build_prompt(transcript: str) -> str:
    msgs = [
        {'role': 'system', 'content': SYSTEM + '\n\n' + FEW_SHOT},
        {'role': 'user', 'content': f'Transcript:\n{transcript}\n\nJSON:'},
    ]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

In [ ]:
SCRIPTS = [
    ('authority+fear', 'This is Agent Miller with the DEA. We have a warrant for your arrest. You need to verify your identity right now or we will dispatch officers.'),
    ('tech_support', 'Hello, this is Microsoft support. We detected a virus on your computer. Please go to the start menu and let me guide you through removal.'),
    ('credential_harvesting', 'This is your bank. To continue, please read me the security code we just texted to your phone.'),
    ('reward_prize', 'Congratulations! You are the lucky winner of a one thousand dollar Walmart gift card. Press 1 now to claim before it expires.'),
    ('impersonation+urgency', 'Grandpa, it is me. I am in jail and need bail money right away. Please do not tell mom.'),
    ('pretexting', 'I am calling on behalf of the HR benefits department. We are updating records and need to confirm your social security number.'),
    ('financial_manipulation', 'Your Amazon order for 799 dollars is being processed. If this was not you, please stay on the line to reverse the charge with a refund.'),
    ('benign', 'Hey, it is Mike. Just calling to confirm dinner on Friday at seven. Let me know if that still works for you.'),
    ('fear_intimidation+authority', 'This is the Social Security Administration. Your number has been suspended due to suspicious activity in Texas.'),
    ('urgency+financial_manipulation', 'Your utility bill is overdue and service will be disconnected in thirty minutes. Pay by card now to avoid disconnection.'),
]
len(SCRIPTS)

In [ ]:
def extract_json_array(text: str):
    text = text.strip()
    m = re.search(r'\[.*\]', text, re.DOTALL)
    if not m: return None
    try: return json.loads(m.group(0))
    except json.JSONDecodeError: return None

def classify(transcript: str) -> dict:
    prompt = build_prompt(transcript)
    inputs = tok(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=400, do_sample=False, pad_token_id=tok.eos_token_id)
    raw = tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    parsed = extract_json_array(raw)
    return {'raw': raw, 'parsed': parsed}

results = []
for tag, script in SCRIPTS:
    r = classify(script)
    results.append({'tag': tag, **r})
    ok = r['parsed'] is not None
    labels = [t.get('label') for t in (r['parsed'] or [])]
    print(f'[{tag}] parse_ok={ok}  labels={labels}')

In [ ]:
parse_ok = sum(1 for r in results if r['parsed'] is not None)
label_ok = sum(
    1 for r in results
    if r['parsed'] is not None
    and all(t.get('label') in TACTIC_TAXONOMY for t in r['parsed'])
)
print(f'valid_json: {parse_ok}/10')
print(f'labels_all_in_taxonomy: {label_ok}/10')
print('acceptance bar: parse_ok >= 9  AND  label_ok >= 9')

## Acceptance checklist

- [x] Qwen2.5-3B-Instruct loads on T4 in fp16 (`device: cuda:0`)
- [x] parse_ok = 10/10 (exceeds >= 9 bar)
- [x] labels_all_in_taxonomy = 10/10
- [ ] Spot-check that evidenceSpans are verbatim substrings — manual review needed

**T1.2 results (Colab T4 run 2026-04-21):**

| Script tag | Got labels | Notes |
| --- | --- | --- |
| authority+fear | authority, pretexting, urgency | missed fear_intimidation |
| tech_support | tech_support, benign | benign is false positive |
| credential_harvesting | pretexting | missed credential_harvesting |
| reward_prize | pretexting, reward_prize, benign | benign is false positive |
| impersonation+urgency | impersonation, urgency | correct |
| pretexting | pretexting | correct |
| financial_manipulation | pretexting, urgency, benign | missed financial_manipulation |
| benign | (empty) | correct — no scam tactics |
| fear_intimidation+authority | authority, pretexting | missed fear_intimidation |
| urgency+financial_manipulation | urgency, pretexting, financial_manipulation | correct |

**Patterns for Phase 3 prompt v2:**
- `pretexting` overused as catch-all — add disambiguation guidance
- `credential_harvesting` and `fear_intimidation` systematically underdetected — add
  targeted few-shot examples for each in prompt v2
- `benign` spuriously added to scam calls — clarify: benign only when NO attack tactics present

T1.2 **PASSED**. Proceed to T2.4 (`promptLibrary.py` + `tacticClassifier.py`).